In [8]:
# ── CELL 1: Environment Setup ─────────────────────────────────────────────
print("🚀 Setting up NeuroAnimate FYP Pipeline environment...")

import subprocess, sys, os

# ── Pre-configure TensorFlow BEFORE it can auto-load and grab both GPUs ───
os.environ["TF_CPP_MIN_LOG_LEVEL"]     = "3"        # silence TF C++ logs
os.environ["TF_ENABLE_ONEDNN_OPTS"]    = "0"        # disable oneDNN
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"    # stop TF from pre-allocating all VRAM

# ── Step 1: Pin NumPy < 2 FIRST ───────────────────────────────────────────
print("Pinning NumPy 1.26.4 ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy==1.26.4", "--force-reinstall"], check=True)
print("✅ NumPy 1.26.4 pinned")

# ── Step 2: Core ML stack (Mistral 7B + SDXL) ────────────────────────────
# peft / transformers / diffusers / accelerate are tightly coupled.
# ⚠️  Do NOT add torch, torchvision, xformers, onnxruntime-gpu here —
#     they bundle CUDA/NCCL .so files that overwrite Kaggle's system NCCL.
!pip install -q -U \
    "peft==0.17.0" \
    "diffusers==0.30.0" \
    "transformers==4.44.2" \
    "accelerate==0.34.2" \
    "safetensors"

# ── Step 3: Vision / animation requirements ───────────────────────────────
# onnxruntime-gpu (NOT plain onnxruntime) — LivePortrait needs GPU ONNX
# onnx==1.16.1    — last version compatible with protobuf 3.x
!pip install -q \
    "numpy==1.26.4" \
    "opencv-python==4.9.0.80" "opencv-contrib-python==4.9.0.80" \
    "onnxruntime-gpu==1.18.0" "onnx==1.16.1" "insightface" \
    "tyro==0.8.5" "lmdb==1.4.1" "pykalman==0.9.7" \
    "albumentations==1.4.10" \
    "scipy==1.13.1" "scikit-image==0.22.0" "scikit-learn==1.4.2" \
    "mediapipe==0.10.9" "imageio==2.33.0" "imageio-ffmpeg==0.4.9" \
    "ffmpeg-python==0.2.0" "einops==0.7.0" "omegaconf==2.3.0" \
    "tqdm" "rich" "pillow>=10.2.0" "gradio>=4.20.0" \
    "huggingface_hub" "yacs" "pyyaml"

# ── Step 4: X-Portrait specific dependencies ──────────────────────────────
# These are required by the X-Portrait diffusion body animation model
!pip install -q \
    "ema_pytorch" "decord" "face_alignment" \
    "kornia" "open_clip_torch" "einops_exts" \
    "rotary_embedding_torch" "entmax" "torchdiffeq"

# ── Step 5: Pin protobuf LAST — this is the critical conflict fix ──────────
# onnx==1.16.1 + onnxruntime-gpu + insightface all work with protobuf 3.20.x
# TensorFlow (auto-loaded by transformers in Kaggle) REQUIRES protobuf < 4.0
# Anything newer breaks TF's MessageFactory.GetPrototype → 5× AttributeError
print("Pinning protobuf 3.20.3 (TF + insightface + onnx compatibility) ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "protobuf==3.20.3", "--force-reinstall"], check=True)
print("✅ protobuf 3.20.3 pinned")

# ── Step 6: Re-pin NumPy — some packages above may re-upgrade it ──────────
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy==1.26.4", "--force-reinstall"], check=True)
print("✅ NumPy 1.26.4 re-confirmed")

# ── Sanity checks ─────────────────────────────────────────────────────────
import torch
print(f"✅ PyTorch  {torch.__version__} | GPUs: {torch.cuda.device_count()} | GPU0: {torch.cuda.get_device_name(0)}")
import numpy as np
print(f"✅ NumPy    {np.__version__}")
import google.protobuf
print(f"✅ protobuf {google.protobuf.__version__}")

print("✅ Cell 1 done — environment ready (X-Portrait deps included)")


🚀 Setting up NeuroAnimate FYP Pipeline environment...
📥 Checking Repository Integrity...
✅ LivePortrait verified
✅ X-Portrait verified
✅ Cell 1 done — repositories are now complete.


In [11]:
# ── CELL 2: LivePortrait (Face) + X-Portrait (3D Body) + ESRGAN ────────────
# Face: LivePortrait with --flag-pasteback (GPU 0)
# Body: X-Portrait diffusion model (GPU 0, sequential after LP VRAM cleared)
# Upscale: Real-ESRGAN 1.5x

import os, subprocess, glob, json, shutil, ctypes, threading, time
import cv2, numpy as np
from PIL import Image
from huggingface_hub import snapshot_download, hf_hub_download

WORK       = "/kaggle/working"
LP_DIR     = "/kaggle/working/LivePortrait"
XP_DIR     = "/kaggle/working/X-Portrait"
ESRGAN_PTH = "/kaggle/working/RealESRGAN_x2plus.pth"
XP_CKPT    = "/kaggle/working/X-Portrait/checkpoint/model_state-415001.th"
XP_CFG     = "/kaggle/working/X-Portrait/config/cldm_v15_appearance_pose_local_mm.yaml"

# ── 1. Clone LivePortrait ────────────────────────────────────────────────
if not os.path.exists(LP_DIR):
    subprocess.run(["git","clone","https://github.com/KwaiVGI/LivePortrait",LP_DIR],check=True)
    print("✅ LivePortrait cloned")
else: print("✅ LivePortrait present")

if not os.path.exists(os.path.join(LP_DIR,"pretrained_weights","liveportrait")):
    os.chdir(LP_DIR)
    snapshot_download(repo_id="KwaiVGI/LivePortrait",local_dir="pretrained_weights",
                      ignore_patterns=["*.git*","README.md","docs/*"])
    print("✅ LivePortrait weights downloaded")
else: print("✅ LivePortrait weights present")

# ── 2. Clone X-Portrait ─────────────────────────────────────────────────
if not os.path.exists(XP_DIR):
    subprocess.run(["git","clone","https://github.com/bytedance/X-Portrait",XP_DIR],check=True)
    print("✅ X-Portrait cloned")
else: print("✅ X-Portrait present")

# ── 3. Install X-Portrait deps ──────────────────────────────────────────
subprocess.run(["pip","install","-q","ema_pytorch","decord","face_alignment",
                "einops","kornia","open_clip_torch","einops_exts",
                "rotary_embedding_torch","entmax","torchdiffeq"],
               check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("✅ X-Portrait deps installed")

# ── 4. Download X-Portrait checkpoint from HuggingFace ───────────────────
os.makedirs(os.path.join(XP_DIR,"checkpoint"), exist_ok=True)
if not os.path.exists(XP_CKPT) or os.path.getsize(XP_CKPT) < 1e9:
    print("⬇️  Downloading X-Portrait checkpoint (~12.3 GB) ...")
    hf_hub_download(repo_id="fffiloni/X-Portrait",
                    filename="model_state-415001.th",
                    local_dir=os.path.join(XP_DIR,"checkpoint"),
                    local_dir_use_symlinks=False)
    for root, dirs, files in os.walk(os.path.join(XP_DIR,"checkpoint")):
        for f in files:
            if f == "model_state-415001.th":
                src = os.path.join(root, f)
                if src != XP_CKPT: shutil.move(src, XP_CKPT)
    if os.path.exists(XP_CKPT):
        print(f"✅ X-Portrait checkpoint ready ({os.path.getsize(XP_CKPT)/1e9:.1f} GB)")
    else:
        raise FileNotFoundError("X-Portrait checkpoint download failed")
else:
    print(f"✅ X-Portrait checkpoint present ({os.path.getsize(XP_CKPT)/1e9:.1f} GB)")

# ── 5. Patch X-Portrait for PyTorch 2.6+ ────────────────────────────────
_XP_PATCHED = os.path.join(XP_DIR, ".torch_patched")
if not os.path.exists(_XP_PATCHED):
    import pathlib
    for pyf in pathlib.Path(XP_DIR).rglob("*.py"):
        txt = pyf.read_text()
        if "torch.load(" in txt and "weights_only" not in txt:
            patched = txt.replace("torch.load(", "torch.load(weights_only=False, ")
            pyf.write_text(patched)
            print(f"   ✅ Patched torch.load in {pyf.relative_to(XP_DIR)}")
    open(_XP_PATCHED,"w").close()
    print("✅ X-Portrait patched for PyTorch 2.6+")
else: print("✅ X-Portrait already patched")

# ── 6. ESRGAN weights ───────────────────────────────────────────────────
if not os.path.exists(ESRGAN_PTH):
    subprocess.run(["wget","-q","-O",ESRGAN_PTH,
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth"],
        check=True, stderr=subprocess.DEVNULL)
    print("✅ ESRGAN weights downloaded")
else: print("✅ ESRGAN weights present")

os.chdir(WORK)
print("✅ Setup complete — LivePortrait + X-Portrait + ESRGAN ready")
# ── CELL 2 (continued): Pipeline Functions ───────────────────────────────

def ffprobe_info(path):
    r = subprocess.run(["ffprobe","-v","quiet","-print_format","json","-show_streams",path],
                       capture_output=True, text=True)
    s = json.loads(r.stdout).get("streams", [])
    return next((x for x in s if x["codec_type"] == "video"), {})

def clear_gpu_memory(log=print):
    log("🧹 Clearing GPU VRAM ...")
    subprocess.run(["python","-c",
        "import torch\n"
        "for i in range(torch.cuda.device_count()):\n"
        "    with torch.cuda.device(i): torch.cuda.empty_cache(); torch.cuda.ipc_collect()\n"
        "    print(f'  GPU {i}: {torch.cuda.memory_allocated(i)/1e9:.2f} GB')\n"],
        capture_output=True, text=True)
    log("✅ GPU VRAM cleared")

# ── Stage 0: Normalise inputs ────────────────────────────────────────────
def normalize_video(source_image_path, driving_video_path, log=print):
    log("📥 Normalising inputs ...")
    src = f"{WORK}/my_image.png"; trm = f"{WORK}/driving_trimmed.mp4"
    shutil.copy(source_image_path, src)
    subprocess.run(["ffmpeg","-y","-i",driving_video_path,
        "-vf","scale=512:768:force_original_aspect_ratio=decrease,"
              "pad=512:768:(ow-iw)/2:(oh-ih)/2:color=black",
        "-r","25","-c:v","libx264","-crf","18","-preset","fast",trm],
        check=True, stderr=subprocess.DEVNULL)
    # 512x512 version for X-Portrait
    trm_sq = f"{WORK}/driving_trimmed_512.mp4"
    subprocess.run(["ffmpeg","-y","-i",driving_video_path,
        "-vf","scale=512:512:force_original_aspect_ratio=decrease,"
              "pad=512:512:(ow-iw)/2:(oh-ih)/2:color=black",
        "-r","25","-c:v","libx264","-crf","18","-preset","fast",trm_sq],
        check=True, stderr=subprocess.DEVNULL)
    src_sq = f"{WORK}/my_image_512.png"
    img = cv2.imread(src)
    cv2.imwrite(src_sq, cv2.resize(img, (512, 512)))
    log("✅ Inputs normalised (512×768 for LP, 512×512 for XP)")
    return src, trm, src_sq, trm_sq

# ── Stage 1a: LivePortrait face-only (GPU 0) ────────────────────────────
def run_liveportrait(source_image, trimmed_video, motion_multiplier=0.65, log=print):
    log(f"🎭 [GPU 0] LivePortrait face (motion×{motion_multiplier}) ...")
    env = os.environ.copy(); env["CUDA_VISIBLE_DEVICES"] = "0"
    _done = f"{LP_DIR}/.lip_zero_patched"
    if not os.path.exists(_done):
        for cfg in [f"{LP_DIR}/src/config/argument_config.py",
                    f"{LP_DIR}/src/config/inference_config.py"]:
            if os.path.exists(cfg):
                with open(cfg,"r") as fh: s = fh.read()
                p = s.replace("flag_lip_zero: bool = True","flag_lip_zero: bool = False"
                   ).replace("flag_lip_zero=True","flag_lip_zero=False")
                if p != s:
                    with open(cfg,"w") as fh: fh.write(p)
        open(_done,"w").close()
    os.chdir(LP_DIR)
    subprocess.run(["python","inference.py","-s",source_image,"-d",trimmed_video,
        "--flag-relative-motion","--flag-pasteback","--flag-do-crop",
        "--driving-multiplier",str(motion_multiplier),"--scale","3.0"],
        check=True, stderr=subprocess.DEVNULL, env=env)
    os.chdir(WORK)
    vids = [v for v in glob.glob(f"{LP_DIR}/animations/**/*.mp4", recursive=True)
            if 'concat' not in os.path.basename(v)]
    lp = max(vids or glob.glob(f"{LP_DIR}/animations/**/*.mp4", recursive=True),
             key=os.path.getmtime)
    log(f"✅ Face animation → {os.path.basename(lp)}")
    return lp

# ── Stage 1b: X-Portrait 3D body (GPU 0, after LP cleared) ──────────────
_XP_WORKER = r'''
import sys,os,torch
XP_DIR=sys.argv[1]; sys.path.insert(0,XP_DIR); os.chdir(XP_DIR)
_ol=torch.load
def _pl(*a,**k): k.setdefault('weights_only',False); return _ol(*a,**k)
torch.load=_pl
from core.test_xportrait import x_portrait_data_prep,load_state_dict,visualize_mm
from model_lib.ControlNet.cldm.model import create_model
cfg=sys.argv[2];ckpt=sys.argv[3];src=sys.argv[4];drv=sys.argv[5]
out=sys.argv[6];steps=int(sys.argv[7]);bf=int(sys.argv[8])
dev=torch.device('cuda:0'); os.makedirs(out,exist_ok=True)
print("[XP] Creating model...",flush=True)
model=create_model(cfg).cpu(); model.sd_locked=True; model.only_mid_control=False
model.to(dev)
print("[XP] Loading checkpoint...",flush=True)
load_state_dict(model,ckpt,strict=False)
print("[XP] Preparing data...",flush=True)
batch=x_portrait_data_prep(src,drv,dev,best_frame_id=bf,output_local=True,target_resolution=512)
batch['video_name']=os.path.basename(drv); batch['source_name']=src
nS=batch['sources'].shape[0]
print(f"[XP] {nS} frames, ddim_steps={steps}",flush=True)
class A: pass
a=A(); a.control_type="appearance_pose_local_mm"; a.control_mode="controlnet_important"
a.uc_scale=5; a.ddim_steps=steps; a.eta=0.0; a.use_fp16=True; a.wonoise=True
a.num_drivings=16; a.device=dev; a.initial_facevid2vid_results=None
visualize_mm(a,"xp_body",batch,model,nSample=nS,local_image_dir=out,num_mix=4)
print("[XP] Done",flush=True)
'''
_XP_WP = "/tmp/xp_worker.py"
with open(_XP_WP, "w") as f: f.write(_XP_WORKER)

def run_xportrait_body(source_512, driving_512, ddim_steps=20, best_frame=0, log=print):
    log(f"🕺 [GPU 0] X-Portrait 3D body animation (ddim={ddim_steps}) ...")
    XP_OUT = f"{WORK}/xp_output"
    if os.path.exists(XP_OUT): shutil.rmtree(XP_OUT)
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "0"
    env["PYTHONPATH"] = XP_DIR + ":" + env.get("PYTHONPATH","")
    res = subprocess.run(
        ["python", _XP_WP, XP_DIR, XP_CFG, XP_CKPT,
         source_512, driving_512, XP_OUT, str(ddim_steps), str(best_frame)],
        capture_output=True, text=True, env=env, timeout=3600)
    for line in res.stdout.strip().split("\n"):
        if line.strip(): log(f"   {line}")
    if res.returncode != 0:
        raise RuntimeError(f"X-Portrait failed:\n{res.stderr[-1500:]}")
    # X-Portrait outputs frames as PNG; assemble into video
    frames = sorted(glob.glob(f"{XP_OUT}/*.png"))
    if not frames:
        # check for mp4 output
        vids = sorted(glob.glob(f"{XP_OUT}/*.mp4"), key=os.path.getmtime)
        if not vids:
            raise FileNotFoundError("X-Portrait produced no output")
        xp_vid = vids[-1]
    else:
        xp_vid = f"{WORK}/xp_body.mp4"
        subprocess.run(["ffmpeg","-y","-framerate","25",
            "-i",f"{XP_OUT}/frame_%05d.png" if os.path.exists(f"{XP_OUT}/frame_00000.png")
                 else f"{XP_OUT}/%05d.png",
            "-c:v","libx264","-crf","18","-pix_fmt","yuv420p",xp_vid],
            check=True, stderr=subprocess.DEVNULL)
    log(f"✅ X-Portrait 3D body → {os.path.basename(xp_vid)} ({os.path.getsize(xp_vid)/1e6:.1f} MB)")
    return xp_vid

# ── Stage 2: Composite LP face + XP body ────────────────────────────────
def composite_videos(lp_video, xp_video, source_image, trimmed_video, log=print):
    log("🎬 Compositing LP face + XP 3D body ...")
    FINAL = f"{WORK}/final_output.mp4"
    bi = ffprobe_info(lp_video)
    BW = int(bi["width"]); BH = int(bi["height"])
    num, den = bi.get("r_frame_rate","25/1").split("/"); FPS_C = float(num)/float(den)

    src_r = cv2.resize(cv2.imread(source_image), (BW, BH))
    gray = cv2.cvtColor(src_r, cv2.COLOR_BGR2GRAY)
    fc = cv2.CascadeClassifier(cv2.data.haarcascades+'haarcascade_frontalface_default.xml')
    SPLIT_Y = int(BH * 0.62)
    for sf, mn, msz in [(1.05,6,(60,60)),(1.05,4,(50,50)),(1.1,3,(40,40))]:
        fcs = fc.detectMultiScale(gray, sf, mn, minSize=msz)
        if len(fcs):
            fx,fy,fw,fh = max(fcs, key=lambda r: r[2]*r[3])
            SPLIT_Y = min(int(fy + fh*1.6), int(BH*0.80)); break
    log(f"   Split y={SPLIT_Y}/{BH}")

    FADE = 80
    lp_mask = np.zeros((BH, BW), dtype=np.float32)
    for y in range(BH):
        if y < SPLIT_Y - FADE: lp_mask[y,:] = 1.0
        elif y < SPLIT_Y + FADE: lp_mask[y,:] = (SPLIT_Y+FADE-y)/(2.0*FADE)
    lp_mask = cv2.GaussianBlur(lp_mask, (1,101), 30)
    lp_m3 = np.stack([lp_mask]*3, axis=-1)
    body_m3 = 1.0 - lp_m3

    def peek(path):
        p = subprocess.Popen(["ffmpeg","-i",path,"-frames:v","1","-f","rawvideo",
            "-pix_fmt","bgr24","-vf",f"scale={BW}:{BH}","-"],
            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
        raw = p.stdout.read(BW*BH*3); p.wait()
        return np.frombuffer(raw,dtype=np.uint8).reshape((BH,BW,3)).astype('float32') if len(raw)==BW*BH*3 else None

    lp0 = peek(lp_video); bp0 = peek(xp_video)
    cc = np.ones(3, dtype='float32')
    if lp0 is not None and bp0 is not None:
        st, sb = max(0,SPLIT_Y-30), min(BH,SPLIT_Y+30)
        for c in range(3):
            lm = float(lp0[st:sb,:,c].mean())+1e-3
            bm = float(bp0[st:sb,:,c].mean())+1e-3
            cc[c] = np.clip(lm/bm, 0.8, 1.25)
        log(f"   Colour-correction gain (BGR): {cc.round(3).tolist()}")

    def opipe(p):
        return subprocess.Popen(["ffmpeg","-i",p,"-f","rawvideo","-pix_fmt","bgr24",
            "-vf",f"scale={BW}:{BH}","-"], stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
    def rframe(pipe):
        raw = pipe.stdout.read(BW*BH*3)
        return np.frombuffer(raw,dtype=np.uint8).reshape((BH,BW,3)).copy() if len(raw)==BW*BH*3 else None

    fp = opipe(lp_video); bp = opipe(xp_video)
    fout = subprocess.Popen(["ffmpeg","-y","-f","rawvideo","-pix_fmt","bgr24",
        "-s",f"{BW}x{BH}","-r",str(FPS_C),"-i","pipe:0",
        "-i",trimmed_video,"-map","0:v","-map","1:a?",
        "-c:v","libx264","-crf","18","-preset","fast","-pix_fmt","yuv420p","-shortest",FINAL],
        stdin=subprocess.PIPE, stderr=subprocess.DEVNULL)
    idx = 0
    while True:
        fb = rframe(fp); bb = rframe(bp)
        if fb is None or bb is None: break
        fb_f = fb.astype('float32')
        bb_f = np.clip(bb.astype('float32')*cc[np.newaxis,np.newaxis,:], 0, 255)
        comp = np.clip(fb_f*lp_m3 + bb_f*body_m3, 0, 255).astype('uint8')
        fout.stdin.write(comp.tobytes()); idx += 1
    fp.stdout.close(); fp.wait(); bp.stdout.close(); bp.wait()
    fout.stdin.close(); fout.wait()
    log(f"✅ Composited {idx} frames → {os.path.getsize(FINAL)/1e6:.1f} MB")
    return FINAL
# ── CELL 2 (continued): ESRGAN + Master Pipeline ────────────────────────

# ── Stage 3: Real-ESRGAN upscale ─────────────────────────────────────────
def auto_batch_size(final_video, target_gb=12.0, log=print):
    info = ffprobe_info(final_video)
    W = int(info.get("width",480)); H = int(info.get("height",640))
    pw = (int(W*3/8))*2; ph = (int(H*3/8))*2
    pixels = pw * ph; K = 1.69e-6; MODEL_GB = 0.5
    vram_per_frame = max(K * pixels, 0.05)
    batch = min(max(1, int((target_gb - MODEL_GB) / vram_per_frame)), 48)
    log(f"   Auto VRAM tune: {W}×{H} → {pw}×{ph} | batch={batch}")
    return batch

_WORKER = """
import sys,os,json,cv2,numpy as np,torch,torch.nn as nn,torch.nn.functional as F
class ResidualDenseBlock(nn.Module):
    def __init__(self,nf=64,gc=32):
        super().__init__()
        self.conv1=nn.Conv2d(nf,gc,3,1,1);self.conv2=nn.Conv2d(nf+gc,gc,3,1,1)
        self.conv3=nn.Conv2d(nf+2*gc,gc,3,1,1);self.conv4=nn.Conv2d(nf+3*gc,gc,3,1,1)
        self.conv5=nn.Conv2d(nf+4*gc,nf,3,1,1);self.lrelu=nn.LeakyReLU(0.2,True)
    def forward(self,x):
        x1=self.lrelu(self.conv1(x));x2=self.lrelu(self.conv2(torch.cat((x,x1),1)))
        x3=self.lrelu(self.conv3(torch.cat((x,x1,x2),1)))
        x4=self.lrelu(self.conv4(torch.cat((x,x1,x2,x3),1)))
        return self.conv5(torch.cat((x,x1,x2,x3,x4),1))*0.2+x
class RRDB(nn.Module):
    def __init__(self,nf,gc=32):
        super().__init__()
        self.rdb1=ResidualDenseBlock(nf,gc);self.rdb2=ResidualDenseBlock(nf,gc)
        self.rdb3=ResidualDenseBlock(nf,gc)
    def forward(self,x): return self.rdb3(self.rdb2(self.rdb1(x)))*0.2+x
class RRDBNet(nn.Module):
    def __init__(self,ic=3,oc=3,nf=64,nb=23,gc=32,scale=2):
        super().__init__()
        self.scale=scale
        self.conv_first=nn.Conv2d(ic*4 if scale==2 else ic,nf,3,1,1)
        self.body=nn.Sequential(*[RRDB(nf,gc) for _ in range(nb)])
        self.conv_body=nn.Conv2d(nf,nf,3,1,1);self.conv_up1=nn.Conv2d(nf,nf,3,1,1)
        self.conv_up2=nn.Conv2d(nf,nf,3,1,1);self.conv_hr=nn.Conv2d(nf,nf,3,1,1)
        self.conv_last=nn.Conv2d(nf,oc,3,1,1);self.lrelu=nn.LeakyReLU(0.2,True)
    def forward(self,x):
        f=F.pixel_unshuffle(x,2) if self.scale==2 else x
        f=self.conv_first(f);f=f+self.conv_body(self.body(f))
        f=self.lrelu(self.conv_up1(F.interpolate(f,scale_factor=2,mode='nearest')))
        f=self.lrelu(self.conv_up2(F.interpolate(f,scale_factor=2,mode='nearest')))
        return self.conv_last(self.lrelu(self.conv_hr(f)))
gpu_id=int(sys.argv[1]);fps=json.loads(sys.argv[2])
fo=sys.argv[3];bs=int(sys.argv[4]);wt=sys.argv[5]
dev=torch.device('cuda:0')
os.makedirs(fo,exist_ok=True)
m=RRDBNet();ck=torch.load(wt,map_location='cpu',weights_only=False)
m.load_state_dict(ck.get('params_ema',ck.get('params',ck)),strict=True)
m.eval().half().to(dev)
def pre(paths):
    imgs=[cv2.cvtColor(cv2.imread(p),cv2.COLOR_BGR2RGB) for p in paths]
    t=[(torch.from_numpy(i).float()/255.0).permute(2,0,1).unsqueeze(0) for i in imgs]
    return torch.cat(t,0).half().to(dev)
def post(t):
    o=t.float().clamp(0,1).cpu().numpy()
    return [cv2.cvtColor((o[i].transpose(1,2,0)*255).astype('uint8'),cv2.COLOR_RGB2BGR) for i in range(o.shape[0])]
total=len(fps)
torch.cuda.reset_peak_memory_stats(dev)
with torch.no_grad():
    for s in range(0,total,bs):
        bp=fps[s:s+bs];out=post(m(pre(bp)))
        for p,img in zip(bp,out): cv2.imwrite(f'{fo}/{os.path.basename(p)}',img)
        done=min(s+bs,total);pk=torch.cuda.max_memory_allocated(dev)/1e9
        print(f'[GPU {gpu_id}] {done}/{total} | VRAM {pk:.1f}GB',flush=True)
print(f'[GPU {gpu_id}] Done. Peak: {torch.cuda.max_memory_allocated(dev)/1e9:.2f}GB',flush=True)
"""
_WP = "/tmp/esrgan_worker.py"
with open(_WP, "w") as f: f.write(_WORKER)

def run_esrgan(final_video, use_dual_gpu=True, log=print):
    log("🔍 Starting Real-ESRGAN 1.5× upscaling ...")
    FI = f"{WORK}/frames_in"; FO = f"{WORK}/frames_esr"
    EV = f"{WORK}/enhanced_output.mp4"
    for d in [FI, FO]:
        if os.path.exists(d): shutil.rmtree(d)
        os.makedirs(d)
    subprocess.run(["ffmpeg","-y","-i",final_video,
                    "-vf","scale=trunc(iw*3/8)*2:trunc(ih*3/8)*2",
                    f"{FI}/frame_%05d.png"], check=True, stderr=subprocess.DEVNULL)
    all_frames = sorted(glob.glob(f"{FI}/frame_*.png")); total = len(all_frames)
    batch_size = auto_batch_size(final_video, target_gb=12.0, log=log)
    import torch; n_gpus = min(torch.cuda.device_count(), 2) if use_dual_gpu else 1
    log(f"   {total} frames | {n_gpus} GPU(s) | batch={batch_size}")
    mid = total // 2
    splits = [all_frames[:mid], all_frames[mid:]] if n_gpus == 2 else [all_frames]
    errors = []
    def worker(gid, fl):
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = str(gid)
        env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
        res = subprocess.run(
            ["python",_WP,str(gid),json.dumps(fl),FO,str(batch_size),ESRGAN_PTH],
            capture_output=True, text=True, env=env)
        for line in res.stdout.strip().split("\n"): log(f"   {line}")
        if res.returncode != 0: errors.append(res.stderr[-600:])
    ts = [threading.Thread(target=worker, args=(i, splits[i])) for i in range(n_gpus)]
    t0 = time.time()
    for t in ts: t.start()
    for t in ts: t.join()
    if errors: raise RuntimeError("ESRGAN failed:\n" + errors[0])
    log(f"✅ Upscaling done in {time.time()-t0:.1f}s")
    all_out = sorted(glob.glob(f"{FO}/frame_*.png"))
    for i, fp in enumerate(all_out): os.rename(fp, f"{FO}/seq_{i:05d}.png")
    subprocess.run(["ffmpeg","-y","-framerate","25","-i",f"{FO}/seq_%05d.png",
                    "-i",f"{WORK}/driving_trimmed.mp4","-map","0:v","-map","1:a?",
                    "-c:v","libx264","-crf","16","-preset","slow",
                    "-pix_fmt","yuv420p","-shortest",EV],
                   check=True, stderr=subprocess.DEVNULL)
    log(f"✅ Enhanced → {os.path.getsize(EV)/1e6:.1f} MB")
    return EV

# ── Master Pipeline ──────────────────────────────────────────────────────
def full_pipeline(source_image_path, driving_video_path,
                  motion_multiplier=0.55, enable_upscale=True,
                  use_dual_gpu=True, log=print):
    log("━"*52)
    log("🚀 NeuroAnimate Pipeline  [LP Face + XP 3D Body]")
    log("━"*52)

    source, trimmed, src_512, drv_512 = normalize_video(
        source_image_path, driving_video_path, log)

    # Stage 1a: LivePortrait face (GPU 0)
    log("⚡ Stage 1a: LivePortrait face animation (GPU 0) ...")
    lp_video = run_liveportrait(source, trimmed, motion_multiplier, log)

    # Clear GPU before X-Portrait (it needs ~12GB VRAM)
    clear_gpu_memory(log)

    # Stage 1b: X-Portrait 3D body (GPU 0)
    log("⚡ Stage 1b: X-Portrait 3D body animation (GPU 0) ...")
    xp_video = run_xportrait_body(src_512, drv_512, ddim_steps=20, log=log)

    clear_gpu_memory(log)

    # Stage 2: Composite face + body
    final = composite_videos(lp_video, xp_video, source, trimmed, log)

    # Stage 3: ESRGAN upscale
    if enable_upscale:
        final = run_esrgan(final, use_dual_gpu=use_dual_gpu, log=log)

    log("━"*52)
    log(f"🎉 Pipeline complete → {final}")
    log("━"*52)
    return final

print("✅ Cell 2 done — NeuroAnimate pipeline ready  [LP Face + XP 3D Body]")
print("   LP face (GPU 0) → clear VRAM → XP body (GPU 0) → Composite → ESRGAN")
print("\n▶ Run Cell 3 to initialise the pipeline class")


🔍 Diagnostic Check:
✅ LivePortrait folder found at: /kaggle/working/LivePortrait
📁 Contents: ['speed.py', 'pretrained_weights', 'requirements.txt', 'requirements_macOS.txt', '.git', 'app.py', 'src', 'app_animals.py', '.gitignore', 'assets', 'inference.py', 'requirements_base.txt', 'LICENSE', 'readme_zh_cn.md', '.vscode', 'inference_animals.py', 'readme.md']
✅ 'src' folder found inside LivePortrait.
❌ Still failing to load modules: No module named 'src'

💡 TIP: If you see 'No module named src', try running Cell 1 again.


In [8]:
# ── CELL 3: Pipeline Class (Mistral 7B + SDXL + NeuroAnimate Video) ────────
print("🔧 Initialising AI Pipeline with GPU optimisation ...")

from transformers import AutoTokenizer, AutoModelForCausalLM
from diffusers import StableDiffusionXLImg2ImgPipeline, StableDiffusionXLPipeline
from IPython.display import display, FileLink, Video
from datetime import datetime
import os, traceback, subprocess, glob, shutil, sys
import torch, gc, math
from PIL import Image
import gradio as gr
import numpy as np
import random
import cv2

# ── GPU clear helper (clears ALL GPUs) ─────────────────────────────────────
def clear_gpu():
    for i in range(torch.cuda.device_count()):
        torch.cuda.set_device(i)
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    gc.collect()

UINT32_MAX = 2 ** 32


class SimplePipeline:
    def __init__(self):
        self.enhanced_prompt = None
        self.images = None

    # ── 1. Prompt Enhancement — Mistral 7B on GPU 1 ───────────────────────
    def enhance(self, user_prompt, mode, enhance_choice):
        if enhance_choice == "No":
            self.enhanced_prompt = user_prompt
            return user_prompt

        try:
            clear_gpu()

            model_name = "NousResearch/Nous-Hermes-2-Mistral-7B-DPO"
            tokenizer  = AutoTokenizer.from_pretrained(model_name)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token

            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float16,
                device_map={"": 1},
                trust_remote_code=True
            ).eval()

            system_prompt_photorealism = (
                "You are an SDXL 1.0 prompt engineer for photorealistic DSLR/Canon images. "
                "Enhance the user prompt into ~52–68 tokens. "
                "Keep the original subject Exactly as given, just enhance it properly. "
                "Must Add: ultra-detailed,camera type (DSLR/mirrorless), lens (mm & f-stop), angle, time of night or day, "
                "lighting, realistic textures, shadows and atmosphere. "
                "Write in 2–3 fluent sentences. "
                "Never use cartoon, anime, painting, illustration styles and storytelling.\n\n"
                f"User Prompt: {user_prompt}\n"
                "Enhanced Prompt:"
            )

            system_prompt_graphic = (
                "You are an elite SDXL prompt enhancer and professional graphic/logo design director. "
                "Rewrite and expand the user's idea into ~40–60 tokens (2–3 short sentences) while keeping the exact subject. "
                "Include: usage contexts, logo variants, vector-friendly scalable geometry & safe spacing , color palette with HEX "
                "3D material + lighting cues, photorealistic mockups, typography pairing, high-contrast"
                "modern timeless aesthetic, SVG-ready composition. "
                "Always use the brand product view\n\n"
                f"User Prompt: {user_prompt}\n"
                "Enhanced Prompt:"
            )

            system_prompt_gaming = (
                "You are an SDXL prompt engineer and AAA game character designer. "
                "Rewrite the user's idea into ~50–60 tokens while keeping the same subject. "
                "Always include: 8k ultra-sharp details,half upper body, Unreal Engine 5 render, RTX lighting, "
                "PBR materials, polygonal hard edges, dark 3D colors, and next-gen shaders. "
                "with tactical armor, sharp metallic textures, detailed weapon. "
                "Focus on sharp edges, high-resolution textures, and the gritty realism of games. "
                "Write in 2–3 fluent sentences emphasizing AAA in-game cutscene style, not movie realism. \n\n"
                f"User Prompt: {user_prompt}\n"
                "Enhanced Prompt:"
            )

            if str(mode).strip().lower() in ('graphic designer', 'graphic', 'designer'):
                system_prompt = system_prompt_graphic
            elif str(mode).strip().lower() in ('gaming', 'game', 'concept artist'):
                system_prompt = system_prompt_gaming
            else:
                system_prompt = system_prompt_photorealism

            inputs = tokenizer(system_prompt, return_tensors="pt",
                               truncation=True, max_length=180)
            inputs = {k: v.to(torch.device("cuda:1")) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=90,
                    temperature=0.7,
                    top_p=0.9,
                    do_sample=True,
                    pad_token_id=tokenizer.eos_token_id
                )

            input_len  = inputs["input_ids"].shape[-1]
            gen_tokens = outputs[0][input_len:]
            enhanced   = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
            enhanced_tokens = enhanced.split()
            if len(enhanced_tokens) > 77:
                enhanced = " ".join(enhanced_tokens[:77])

            self.enhanced_prompt = enhanced
            del model, tokenizer, inputs, outputs
            clear_gpu()
            return enhanced

        except Exception as e:
            clear_gpu()
            return f"❌ Error in enhancement: {str(e)}"

    # ── 2. Image Generation — SDXL Base (GPU 0) + Refiner (GPU 1) ─────────
    def generate_images(self, final_prompt, num_images, seed):
        try:
            clear_gpu()

            torch.cuda.set_device(0)
            pipe_base = StableDiffusionXLPipeline.from_pretrained(
                "stabilityai/stable-diffusion-xl-base-1.0",
                torch_dtype=torch.float16,
                variant="fp16",
                use_safetensors=True
            ).to("cuda:0")

            base_images = []
            for i in range(num_images):
                generator = torch.Generator("cuda:0").manual_seed(seed + i)
                image = pipe_base(
                    final_prompt,
                    num_inference_steps=35,
                    guidance_scale=8,
                    height=768,
                    width=768,
                    generator=generator
                ).images[0]
                base_images.append(image)

            del pipe_base
            clear_gpu()

            torch.cuda.set_device(1)
            refiner = StableDiffusionXLImg2ImgPipeline.from_pretrained(
                "stabilityai/stable-diffusion-xl-refiner-1.0",
                torch_dtype=torch.float16,
                variant="fp16",
                use_safetensors=True
            ).to("cuda:1")

            refined_images = []
            for img in base_images:
                refined = refiner(
                    prompt=final_prompt,
                    image=img,
                    num_inference_steps=105,
                    guidance_scale=8,
                    strength=0.25
                ).images[0]
                refined_images.append(refined)
                os.makedirs("refined_images", exist_ok=True)
                filename = (f"refined_images/img_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
                            f"_{len(refined_images)}.png")
                refined.convert("RGB").save(filename)

            del refiner
            clear_gpu()

            self.images = refined_images

            if len(refined_images) > 1:
                cols  = math.ceil(math.sqrt(len(refined_images)))
                rows  = math.ceil(len(refined_images) / cols)
                width, height = refined_images[0].size
                grid  = Image.new("RGB", (cols * width, rows * height))
                for i, img in enumerate(refined_images):
                    grid.paste(img, (i % cols * width, i // cols * height))
                return [grid] + refined_images
            else:
                return refined_images

        except Exception as e:
            clear_gpu()
            return [f"❌ Error in image generation: {str(e)}"]

    # ── 3. Video Generation — LP Face + X-Portrait 3D Body + ESRGAN ──────
    def generate_video(self, driving_video_path=None):
        """
        Animates the most recently generated portrait:
          normalize → LivePortrait face (GPU 0) → clear VRAM
          → X-Portrait 3D body (GPU 0) → clear VRAM
          → composite face+body → Real-ESRGAN upscale
        """
        try:
            portrait_files = glob.glob('/kaggle/working/refined_images/*.png')
            if not portrait_files:
                return None, "❌ No generated images found. Please generate images first."
            source_image = max(portrait_files, key=os.path.getmtime)

            if driving_video_path and os.path.exists(driving_video_path):
                driving_video = driving_video_path
            else:
                driving_video = "/kaggle/input/sharukh/shahrukh.mp4.mp4"
                if not os.path.exists(driving_video):
                    return None, ("❌ No driving video found. "
                                  "Please upload a driving video in the sidebar.")

            clear_gpu()

            final_video = full_pipeline(
                source_image_path=source_image,
                driving_video_path=driving_video,
                motion_multiplier=0.55,
                enable_upscale=True,
                use_dual_gpu=True,
                log=print
            )

            if final_video and os.path.exists(final_video):
                size_mb = os.path.getsize(final_video) / 1e6
                return final_video, f"✅ Video generated! ({size_mb:.1f} MB)"
            else:
                return None, "❌ Pipeline returned no output file."

        except Exception as e:
            clear_gpu()
            return None, f"❌ Video generation failed:\n{traceback.format_exc()}"


pipeline = SimplePipeline()
print("✅ Cell 3 done — SimplePipeline initialised!")
print("   enhance() → GPU 1 (Mistral 7B)")
print("   generate_images() → GPU 0 (SDXL Base) + GPU 1 (Refiner)")
print("   generate_video() → LP Face + X-Portrait 3D Body + ESRGAN")
print("\n▶ Run Cell 4 to launch Gradio UI")


🔧 Initialising AI Pipeline with GPU optimisation ...
✅ Cell 3 done — SimplePipeline initialised!
   enhance() → GPU 1 (Mistral 7B)
   generate_images() → GPU 0 (SDXL Base) + GPU 1 (Refiner) → full GPU clear
   generate_video() → NeuroAnimate (LP ∥ Body → Hair → Lip → ESRGAN)

▶ Run Cell 4 to launch Gradio UI


In [ ]:
# ── CELL 4: Gradio UI ─────────────────────────────────────────────────────
print("Setting up Gradio Interface ...")

import gradio as gr, random, threading, time, traceback

with gr.Blocks(theme=gr.themes.Soft(), title="NeuroAnimate FYP Pipeline") as demo:

    gr.Markdown("# 🧠 NeuroAnimate: Text → Image → Animated Video")
    gr.Markdown(
        "**Flow:** Enhance Prompt (Mistral 7B · GPU 1) → "
        "Generate Image (SDXL · GPUs 0+1) → "
        "Animate (LivePortrait Face + X-Portrait 3D Body + ESRGAN)"
    )

    with gr.Row():

        # ── Left sidebar ──────────────────────────────────────────────────
        with gr.Column(scale=1):
            gr.Markdown("### ⚙️ Settings")
            mode_choice = gr.Radio(
                choices=["Photorealism", "Graphic Designer", "Gaming"],
                value="Photorealism", label="Style"
            )
            enhance_choice = gr.Radio(
                choices=["Yes", "No"], value="Yes",
                label="Enhance Prompt?",
                info="'No' uses your original prompt directly"
            )
            num_images     = gr.Slider(1, 4, value=2, step=1,
                                       label="Number of Images")
            use_fixed_seed = gr.Checkbox(value=True, label="Fixed Seed")
            manual_seed    = gr.Number(value=42, label="Seed", precision=0)

            gr.Markdown("---")
            gr.Markdown("### 🎬 Driving Video")
            video_upload = gr.Video(
                label="Upload Driving Video (required for animation)",
                sources=["upload"], interactive=True
            )

            gr.Markdown("---")
            with gr.Row():
                reset_btn   = gr.Button("🔄 Reset")
                enhance_btn = gr.Button("✨ Enhance Prompt", variant="primary")

            generate_images_btn = gr.Button("🖼️ Generate Images",
                                            variant="primary", visible=False)
            generate_video_btn  = gr.Button("🎭 Animate Portrait (3D Body)",
                                            variant="primary", visible=False)

        # ── Main panel ────────────────────────────────────────────────────
        with gr.Column(scale=2):
            user_prompt = gr.Textbox(
                lines=3,
                placeholder="Describe your portrait / subject ...",
                label="Your Prompt"
            )
            enhanced_prompt = gr.Textbox(
                lines=3, label="Enhanced Prompt", interactive=True
            )
            image_gallery = gr.Gallery(
                label="Generated Images", columns=2, height="auto"
            )
            video_output = gr.Video(label="🎥 Animated Video (3D Body)", height=480)

    status_text = gr.Textbox(
        label="Status",
        value="✅ Ready — choose a style, enter a prompt, and click Enhance Prompt.",
        interactive=False
    )

    # ── Callbacks ─────────────────────────────────────────────────────────

    def enhance_click(user_prompt, mode_choice, enhance_choice):
        enhanced   = pipeline.enhance(user_prompt, mode_choice, enhance_choice)
        status_msg = ("✅ Using original prompt. Ready to generate images."
                      if enhance_choice == "No"
                      else "✅ Prompt enhanced! Ready to generate images.")
        return enhanced, gr.update(visible=True), status_msg

    def generate_click(enhanced_prompt, num_images, use_fixed_seed, manual_seed):
        seed   = int(manual_seed) if use_fixed_seed else random.randint(0, 2**32-1)
        images = pipeline.generate_images(enhanced_prompt, int(num_images), seed)
        return (images, gr.update(visible=True),
                "✅ Images generated! Upload a driving video and click Animate Portrait.")

    def video_click(uploaded_video):
        """Streams elapsed time to the UI while the pipeline runs in background."""
        if uploaded_video is None:
            yield None, "❌ Please upload a driving video first."
            return

        result_h = [None]
        err_h    = [None]

        def _run():
            try:
                result_h[0] = pipeline.generate_video(uploaded_video)
            except Exception:
                err_h[0] = traceback.format_exc()

        t = threading.Thread(target=_run, daemon=True)
        t.start()
        ts = time.time()

        while t.is_alive():
            time.sleep(1.5)
            yield None, f"⏱  Animating (LP Face + XP 3D Body)…  {int(time.time()-ts)}s elapsed"

        t.join()

        if err_h[0]:
            yield None, f"❌ Error:\n{err_h[0][:500]}"
            return

        video_result, status = result_h[0]
        yield video_result, status

    def reset_click():
        pipeline.enhanced_prompt = None
        pipeline.images = None
        clear_gpu()
        return (None, None, None, None,
                gr.update(visible=False), gr.update(visible=False),
                "Yes", None, "🔄 Reset complete!")

    # ── Event wiring ──────────────────────────────────────────────────────

    enhance_btn.click(
        enhance_click,
        inputs=[user_prompt, mode_choice, enhance_choice],
        outputs=[enhanced_prompt, generate_images_btn, status_text]
    )
    generate_images_btn.click(
        generate_click,
        inputs=[enhanced_prompt, num_images, use_fixed_seed, manual_seed],
        outputs=[image_gallery, generate_video_btn, status_text]
    )
    generate_video_btn.click(
        video_click,
        inputs=[video_upload],
        outputs=[video_output, status_text]
    )
    reset_btn.click(
        reset_click,
        inputs=[],
        outputs=[user_prompt, enhanced_prompt, image_gallery, video_output,
                 generate_images_btn, generate_video_btn, enhance_choice,
                 video_upload, status_text]
    )

print("Launching Gradio UI ...")
demo.launch(share=True, debug=True)


Setting up Gradio Interface ...
Launching Gradio UI ...
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://313065db7cd91a0e0b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (78 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: [',']
Token indices sequence length is longer than the specified maximum sequence length for this model (78 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: [',']


  0%|          | 0/35 [00:00<?, ?it/s]

...

Token indices sequence length is longer than the specified maximum sequence length for this model (78 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: [',']


  0%|          | 0/26 [00:00<?, ?it/s]

The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: [',']


  0%|          | 0/26 [00:00<?, ?it/s]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🚀 NeuroAnimate Pipeline  [LivePortrait Full-Body]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📥 Normalising inputs ...
✅ Driving video ready — 399 frames @ 512×768 25fps
⚡ Stage 1: LivePortrait full-body (GPU 0) ...
🎭 [GPU 0] Running LivePortrait FULL-BODY (motion×0.55) ...
[06:51:31] Load appearance_feature_extractor from    live_portrait_wrapper.py:46
           /kaggle/working/LivePortrait/pretrained_w                            
           eights/liveportrait/base_models/appearanc                            
           e_feature_extractor.pth done.                                        
[06:51:32] Load motion_extractor from                live_portrait_wrapper.py:49
           /kaggle/working/LivePortrait/pretrained_w                            
           eights/liveportrait/base_models/motion_ex                            
           tractor.pth done.                                                    
[06:51:33] Load